# Building Your First LLM Helper: From Prompt to Inspectable Python Code

In this lesson, we build a small helper that takes a question about a dataset and returns the result. 
Instead of writing analysis code each time, we pass the dataset context and the question to the LLM and let it generate the answer.

## 1 — Setup

We install the necessary libraries, load our API credentials from a `.env` file, and read in the dataset. The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.

In [ ]:
%pip install -qq google-genai pandas scikit-learn matplotlib seaborn python-dotenv


In [ ]:
import os
import re
from google import genai
from dotenv import load_dotenv
import pandas as pd

# Load environment variables from a .env file into the environment
load_dotenv()

# Read the Gemini API key from the environment variables
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Create a Gemini client using the API key
# This client will be used to send requests to Gemini models
client = genai.Client(api_key=GEMINI_API_KEY)


In [ ]:
df = pd.read_csv("../../data/hr_analytics.csv")
print(df.shape)


In [ ]:
df.head().T

## 2 — A Glimpse of the LLM Helper Function


In [ ]:
SYSTEM_PROMPT = (
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Store the final result in `result_df`. "
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)


def eda_helper(question: str, frame: pd.DataFrame):
    """Ask a plain-English question about df and get back a DataFrame result."""

    # Build the prompt sent to the model.
    # We include the column names so the model knows the schema of the dataset.
    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    # Send the prompt and system instructions to the Gemini model.
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,  # Makes the output deterministic
            "seed": 42,
            "system_instruction": SYSTEM_PROMPT,  # System level instructions that define rules or behavior for the model
        },
    )
    # Extract executable Python from the model response.
    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    # Define a restricted execution environment.
    # The generated code can only access pandas (pd) and the DataFrame (df).
    env = {"pd": pd, "df": frame.copy()}

    # Execute the generated code in the restricted environment.
    exec(code, env, env)

    # The model is instructed to store the final output in `result_df`.
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

## 3 — Let's Try It Out

In [ ]:
eda_helper("How many missing values does each column have?", df)

In [ ]:
eda_helper("What is the average age by department?", df)

In [ ]:
eda_helper("What are the top 10 highest paid employees?", df)

## 4 — Show me the Code

Let's add a `show_code` flag so you can see exactly what the LLM wrote before it runs.

In [ ]:
def eda_helper(question, frame: pd.DataFrame, show_code: bool = False):
    """Ask a plain-English question about df; get back a DataFrame."""
    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": SYSTEM_PROMPT,
        },
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    env = {"pd": pd, "df": frame.copy()}
    exec(code, env, env)
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

In [ ]:
eda_helper("How many missing values does each column have?", df, show_code=True)

In [ ]:
eda_helper("What is the average age by department?", df, show_code=True)

In [ ]:
eda_helper("What are the top 10 highest paid employees?", df, show_code=True)